# Figures 4L, 5 and S15: most-affected genes of the Figure 4K double-KO perturbations

Rows = the perturbations shown in the `Fig3_interaction_heatmap` plots (the e1 double
KOs — enriched in the differentiated states + NEUROG1+SIM1), **day04 and day10
separately**. Columns = the genes **most affected** by those perturbations.

The β / FDR matrices come from the **precomputed per-perturbation DE (pdex) tables**
`Src/ComboScreen/Day0{4,10}DEGs_fdr8000.csv` (each `target` perturbation vs `NTC`):
**β = log2(fold_change)** vs NTC, and an **FDR re-corrected for an ~8000-gene test
universe** (`_add_fdr8000.py` re-ran BH per perturbation with the number of tests fixed
at 8000 instead of ~23k — a realistic tested-gene count, less over-penalizing than the
genome-wide correction). The notebook uses this `fdr` directly to pick the genes to plot.

In [ ]:
import _figutils as fu
import importlib; importlib.reload(fu)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

fu.set_theme()

# ---- tunable parameters (play with these) -----------------------------------
SIG_FDR = 0.1        # a gene is "significant" in a perturbation if FDR < SIG_FDR
MIN_SIG_PERTS = 2    # keep genes significant in AT LEAST this many perturbations
TOP_GENES = 60       # number of top genes to show (ranked by #combos significant, then FDR)
# -----------------------------------------------------------------------------
ENR_FDR = 0.1        # differentiated-state enrichment cutoff (defines the e1 set; matches Fig2)
FORCE_GENES = ["CDKN1A"]   # genes always shown as a column (FDR-zeroed like every other gene)

# pdex DE tables with FDR recomputed for an ~8000-gene test universe (built by
# _add_fdr8000.py; the file's `fdr` column already holds these values).
DE_PATH = {
    "day04": "/home/eraslab1/Projects/AbbasScreen/Src/ComboScreen/Day04DEGs_fdr8000.csv",
    "day10": "/home/eraslab1/Projects/AbbasScreen/Src/ComboScreen/Day10DEGs_fdr8000.csv",
}

# Senescence / SASP / cell-cycle-arrest genes to HIGHLIGHT and always keep if significant.
SENESCENCE_GENES = {
    "CDKN1A", "CDKN1B", "CDKN2A", "CDKN2B", "CDKN2C", "CDKN2D", "TP53", "RB1", "RBL2",
    "GADD45A", "GADD45B", "LMNB1", "GLB1", "HMGB1", "HMGB2", "CCNG2", "ELAVL1", "MTAP",
    "FZR1", "SENP1", "SENP2", "SERPINE1", "IGFBP3", "IGFBP7", "IGFBP4", "TFPI2",
    "IL6", "IL1A", "IL1B", "CXCL8", "CXCL1", "CXCL2", "CCL2", "MMP1", "MMP3", "MMP10",
    "TIMP1", "TIMP2", "PLAU", "PLAUR", "ICAM1",
    # strong senescence markers/regulators from the Doxo1 signature
    "GATA4", "B2M", "CCND1", "HMGA2", "HMGA1P4", "MICA", "MICB", "MIR34AHG",
}

# e1 combos per day = enriched in the differentiated states + NEUROG1+SIM1
# (read from Fig2's saved enrichment table; run Fig2's selection cell first).
def e1_combos(day):
    enr = pd.read_csv(fu.FIG_DIR / f"Fig2_state_enrichment_{day}.csv")
    diff = enr[(enr.state.isin(fu.DIFFERENTIATED_STATES)) &
               (enr.odds_ratio > 1) & (enr.fdr < ENR_FDR)]
    return [p for p in (set(diff["perturbation"]) | {"NEUROG1+SIM1"}) if "+" in p]

## Per-day perturbation → gene effect heatmaps (clustered)

For each day: read the pdex DE table (β = log2 fold-change vs NTC, the 8000-test-corrected
FDR). **Two knobs** (set in the cell above): **`MIN_SIG_PERTS`** = the minimum number of
perturbations a gene must be significant (FDR<`SIG_FDR`) in to be a candidate, and
**`TOP_GENES`** = how many of those to show (ranked by #combos significant, then smallest
FDR). On top of that, **any senescence gene significant in ≥1 perturbation is always kept**
(independent of both knobs), **plus CDKN1A** (force-included as a column). **Every gene —
including the forced CDKN1A — is FDR-zeroed** (cells with FDR ≥ `SIG_FDR` → 0). Rows/columns
clustered with **Ward.D2**; senescence genes **highlighted** (red strip + red bold labels).

In [ ]:
for day in fu.TIMEPOINT_ORDER:
    # beta = log2 fold-change vs NTC, and the file's FDR (corrected for an ~8000-gene
    # test universe) for the Fig3 combo rows (chunked, low-memory read).
    combos = sorted(e1_combos(day))
    beta, pval, fdr = fu.load_perturbation_de(DE_PATH[day], day, perts=combos)
    present = [c for c in combos if c in beta.index]
    B, Q = beta.loc[present], fdr.loc[present]
    B.to_csv(fu.FIG_DIR / f"Fig4_perturbation_gene_beta_{day}.csv")
    Q.to_csv(fu.FIG_DIR / f"Fig4_perturbation_gene_fdr_{day}.csv")

    sig = Q < SIG_FDR
    nsig = sig.sum(axis=0)                       # # combos each gene is significant in

    # candidate genes: significant in >= MIN_SIG_PERTS perturbations; rank by #combos
    # significant in (desc), tie-broken by smallest FDR across those combos (asc); cap to TOP_GENES.
    cand = nsig.index[nsig >= MIN_SIG_PERTS]
    score = pd.DataFrame({"n": nsig[cand],
                          "q": Q[cand].where(sig[cand]).min(axis=0)}).sort_values(
                          ["n", "q"], ascending=[False, True])
    top = score.head(TOP_GENES).index.tolist()

    # always filter in any senescence gene significant (FDR<SIG_FDR) in >=1 combo,
    # regardless of MIN_SIG_PERTS / the TOP_GENES cap.
    sen_sig = [g for g in SENESCENCE_GENES if g in sig.columns and sig[g].sum() >= 1]
    sen_keep = [g for g in sen_sig if g not in top]

    # force-include CDKN1A etc. as a column. EVERY gene -- including the forced ones --
    # is FDR-zeroed: cells with FDR >= SIG_FDR are set to 0.
    force = [g for g in FORCE_GENES if g in B.columns and g not in top + sen_keep]
    genes = top + sen_keep + force
    M = B[genes].where(sig[genes], 0.0).fillna(0.0)
    sen_in = [g for g in genes if g in SENESCENCE_GENES]
    print(f"{day}: {len(cand)} genes sig in >={MIN_SIG_PERTS} combos -> showing top {len(top)} "
          f"+ {len(sen_keep)} senescence + forced {force} = {len(genes)} columns | "
          f"senescence in panel: {sen_in}")
    if len(genes) < 2:
        print(f"  too few genes to cluster -- skipping {day}"); continue
    M.to_csv(fu.FIG_DIR / f"Fig4_top_gene_effects_{day}.csv")

    # column colour strip: highlight senescence-related genes
    col_colors = pd.Series({g: ("#d62728" if g in SENESCENCE_GENES else "#f0f0f0") for g in genes},
                           name="senescence")

    # cluster rows (perturbations) and columns (genes) with Ward.D2 (ward2) linkage
    vmax = np.nanmax(np.abs(M.values)) or 1.0
    g = sns.clustermap(M, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
                       metric="euclidean", method="ward", col_colors=col_colors,
                       figsize=(12,6),
                       linewidths=0.2, linecolor="white",
                       cbar_kws={"label": "log2 fold-change vs NTC; 0 if n.s."},
                       xticklabels=True, yticklabels=True)
    # colour the senescence gene tick-labels (red, bold) so they stand out
    for t in g.ax_heatmap.get_xticklabels():
        if t.get_text() in SENESCENCE_GENES:
            t.set_color("#d62728"); t.set_fontweight("bold")
    g.ax_heatmap.set_xlabel("Gene  (red = senescence-related)"); g.ax_heatmap.set_ylabel("Double-KO pair")
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90, fontsize=6)
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0, fontsize=8)
    g.fig.suptitle(f"Fig3 double KOs × top {len(top)} genes sig in >={MIN_SIG_PERTS} perturbations "
                   f"(+ sig senescence + CDKN1A) — {day}\n"
                   f"(pdex DE; FDR over ~8000 genes; non-sig log2FC=0; Ward.D2; red = senescence)",
                   fontsize=9)
    fu.savefig(f"geneeffects_perturbation_gene_heatmap_{day}", g.fig, tight=False)

## Interaction models on the displayed senescence genes (day10)

For the senescence genes shown in the day10 heatmap, re-run the **same statsmodels
interaction decomposition used in Fig3** (`fu.interaction_doxo1_ols`) — but with each
senescence gene's **expression** as the outcome — for the day10 double KOs:

    decomposition:  expr ~ g1 + g2 + g1:g2   (NTC + g1/g2-only cells)  -> individual β_g1, β_g2 + interaction
    total:          expr ~ combo             (NTC vs the double-KO)    -> total double-KO effect

fit on all day10 cells. Output is **one heatmap per perturbation** (`Fig4_senescence_interaction_<combo>_day10`):
rows = senescence genes (ordered by total effect), columns = the **individual** effect of
each constituent gene, the **interaction**, and the **total** double-KO effect; annotated
with the coefficient and significance stars (`*` FDR<`SIG_FDR`). *Loads the processed object
and fits one model per gene, so it takes a few minutes.*

In [ ]:
import scipy.sparse as sp
DAY = "day10"
#combos = sorted(e1_combos(DAY))
combos=['NEUROG1+SIM1']

# senescence genes displayed in the day10 heatmap = senescence genes significant (FDR<SIG_FDR)
# in >=1 day10 combo (same rule the heatmap uses to keep senescence genes).
_, _, fdrP = fu.load_perturbation_de(DE_PATH[DAY], DAY, perts=combos)
presentP = [c for c in combos if c in fdrP.index]
sigP = fdrP.loc[presentP] < SIG_FDR
sen_display = sorted([g for g in SENESCENCE_GENES if g in sigP.columns and sigP[g].sum() >= 1])

# expression data (heavy: loads the processed object)
adata = fu.load("processed")
sen_genes = [g for g in sen_display if g in adata.var_names]
missing = [g for g in sen_display if g not in adata.var_names]
print(f"day10 displayed senescence genes: {len(sen_display)}; modeled: {len(sen_genes)}"
      + (f"; not in expression data: {missing}" if missing else ""))

# per senescence gene: the Fig3 statsmodels interaction decomposition + separate total
# model, with that gene's expression as the outcome, for the day10 combos (all day10 cells).
res = {}
for gene in sen_genes:
    x = adata[:, gene].X
    adata.obs["_yexpr"] = x.toarray().ravel() if sp.issparse(x) else np.asarray(x).ravel()
    res[gene] = fu.interaction_doxo1_ols(adata, combos, day=DAY, score="_yexpr",
                                         states=fu.STATE_ORDER, min_cells=5).set_index("pair")
adata.obs.drop(columns=["_yexpr"], inplace=True, errors="ignore")

def _star(p):
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < SIG_FDR else ""

# ONE heatmap per perturbation: rows = senescence genes, columns = individual effect of
# gene 1, individual effect of gene 2, the interaction, and the total double-KO effect.
beta_cols = ["beta_g1", "beta_g2", "beta_interaction", "total_effect"]
fdr_cols = ["fdr_g1", "fdr_g2", "fdr_interaction", "fdr_total"]
for combo in combos:
    g1, g2 = combo.split("+")[:2]
    B = pd.DataFrame({bc: [res[g].loc[combo, bc] for g in sen_genes] for bc in beta_cols}, index=sen_genes)
    F = pd.DataFrame({fc: [res[g].loc[combo, fc] for g in sen_genes] for fc in fdr_cols}, index=sen_genes)
    if B["total_effect"].notna().sum() == 0:
        print(f"{combo}: not enough cells -- skipped"); continue
    order = B["total_effect"].sort_values(ascending=False, na_position="last").index   # strongest total at top
    B, F = B.loc[order], F.loc[order]
    colnames = [f"{g1}\n(individual)", f"{g2}\n(individual)", "Interaction", "Total\n(double KO)"]
    mat = B.values
    annot = np.array([[f"{mat[i, j]:.3f}{_star(F.values[i, j])}" for j in range(mat.shape[1])]
                      for i in range(mat.shape[0])], dtype=object)
    vmax = np.nanmax(np.abs(mat)) or 1.0
    fig, ax = plt.subplots(figsize=(4.4, max(3, len(sen_genes) * 0.3 + 1)))
    sns.heatmap(pd.DataFrame(mat, index=B.index, columns=colnames), cmap="RdBu_r", center=0,
                vmin=-vmax, vmax=vmax, annot=annot, fmt="", linewidths=0.4, linecolor="white",
                cbar_kws={"label": "effect on log-expr"}, ax=ax, annot_kws={"fontsize": 6})
    ax.axvline(2, color="0.4", lw=1); ax.axvline(3, color="black", lw=2)   # individual | interaction | total
    ax.set_title(f"{combo} — effects on senescence genes (day10)\n(* FDR<{SIG_FDR}, ** <0.01, *** <0.001)",
                 fontsize=9)
    ax.set_ylabel("Senescence gene"); ax.set_xlabel("")
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=6)
    plt.setp(ax.get_xticklabels(), rotation=0, fontsize=8)
    fu.savefig(f"geneeffects_senescence_interaction_{combo}_{DAY}", fig)